## **Week 6 – Feature Engineering and Market Metrics**

In [ ]:
# Price Ratio:                    ClosePrice / OriginalListPrice
# Price Per Sq Ft:                ClosePrice / LivingArea
# Days on Market:                 DaysOnMarket (raw field)
# Year / Month / YrMo:            Derived from CloseDate
# Close to Original List Ratio:   ClosePrice / OriginalListPrice
# Listing to Contract Days:       PurchaseContractDate - ListingContractDate
# Contract to Close Days:         CloseDate - PurchaseContractDate

In [1]:
import pandas as pd

# Load cleaned datasets
sold = pd.read_csv('../data/02_intermediate/CRMLSSold_Cleaned.csv', low_memory=False)
listings = pd.read_csv('../data/02_intermediate/CRMLSListing_Cleaned.csv', low_memory=False)

print(f"Sold: {sold.shape[0]:,} rows, {sold.shape[1]} columns")
print(f"Listings: {listings.shape[0]:,} rows, {listings.shape[1]} columns")

Sold: 421,589 rows, 76 columns
Listings: 467,728 rows, 69 columns


In [2]:
# Convert date fields to datetime
date_fields = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
for field in date_fields:
    sold[field] = pd.to_datetime(sold[field])
    listings[field] = pd.to_datetime(listings[field])

print("Date fields converted successfully")

Date fields converted successfully


In [3]:
# Feature Engineering — Sold Dataset

# 1. Price Ratio — measures negotiation strength (ClosePrice vs ListPrice)
sold['price_ratio'] = sold['ClosePrice'] / sold['ListPrice']

# 2. Close to Original List Ratio — captures full price reduction history
sold['close_to_original_list_ratio'] = sold['ClosePrice'] / sold['OriginalListPrice']

# 3. Price Per Square Foot — normalizes price across different home sizes
sold['price_per_sqft'] = sold['ClosePrice'] / sold['LivingArea']

# 4. Listing to Contract Days — time from listing to accepted offer
sold['listing_to_contract_days'] = (
    sold['PurchaseContractDate'] - sold['ListingContractDate']
).dt.days

# 5. Contract to Close Days — escrow and closing period duration
sold['contract_to_close_days'] = (
    sold['CloseDate'] - sold['PurchaseContractDate']
).dt.days

# 6. Time series variables derived from CloseDate
sold['close_year'] = sold['CloseDate'].dt.year
sold['close_month'] = sold['CloseDate'].dt.month
sold['close_yrmo'] = sold['CloseDate'].dt.to_period('M')

# Sample output table showing new columns populated
print("Sample of engineered metrics:")
sold[['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea',
      'price_ratio', 'close_to_original_list_ratio', 'price_per_sqft',
      'listing_to_contract_days', 'contract_to_close_days',
      'close_year', 'close_month', 'close_yrmo']].head(10)

Sample of engineered metrics:


,ClosePrice,ListPrice,OriginalListPrice,LivingArea,price_ratio,close_to_original_list_ratio,price_per_sqft,listing_to_contract_days,contract_to_close_days,close_year,close_month,close_yrmo
0,5000000.0,5000000.0,5000000.0,4354.0,1.000000,1.000000,1148.369316,0.0,63.0,2024,1,2024-01
1,858000.0,858000.0,NaN,1995.0,1.000000,NaN,430.075188,0.0,0.0,2024,1,2024-01
2,1890500.0,1890500.0,1890500.0,3194.0,1.000000,1.000000,591.891046,0.0,0.0,2024,1,2024-01
3,2100000.0,2100000.0,2100000.0,3736.0,1.000000,1.000000,562.098501,0.0,48.0,2024,1,2024-01
4,1950000.0,1950000.0,1950000.0,2100.0,1.000000,1.000000,928.571429,32.0,34.0,2024,1,2024-01
5,2340000.0,2340000.0,NaN,2442.0,1.000000,NaN,958.230958,0.0,0.0,2024,1,2024-01
6,1485000.0,1550000.0,1550000.0,1601.0,0.958065,0.958065,927.545284,1.0,21.0,2024,1,2024-01
7,1130000.0,999000.0,999000.0,2136.0,1.131131,1.131131,529.026217,1.0,17.0,2024,1,2024-01
8,1060000.0,1050000.0,1050000.0,1917.0,1.009524,1.009524,552.947314,34.0,6.0,2024,1,2024-01
9,2150000.0,2150000.0,NaN,2200.0,1.000000,NaN,977.272727,0.0,0.0,2024,1,2024-01


In [4]:
# Segment Analysis — by CountyOrParish
print("Summary by CountyOrParish (top 15 by transaction count):")
county_summary = sold.groupby('CountyOrParish').agg(
    transaction_count=('ClosePrice', 'count'),
    median_close_price=('ClosePrice', 'median'),
    median_price_per_sqft=('price_per_sqft', 'median'),
    median_days_on_market=('DaysOnMarket', 'median'),
    avg_price_ratio=('price_ratio', 'mean')
).sort_values('transaction_count', ascending=False).head(15)

print(county_summary.to_string())

Summary by CountyOrParish (top 15 by transaction count):
                 transaction_count  median_close_price  median_price_per_sqft  median_days_on_market  avg_price_ratio
CountyOrParish                                                                                                       
Los Angeles                 105191            900000.0             608.073928                   19.0         1.003287
Riverside                    59491            600000.0             320.930674                   30.0         0.990282
San Diego                    54082            890000.0             589.743590                   15.0         1.069388
Orange                       47332           1175000.0             672.214044                   14.0         0.999336
San Bernardino               39654            532834.0             331.991952                   25.0         0.993880
Alameda                      19792           1120000.0             697.475328                   14.0         1.061928

In [5]:
# Segment Analysis — by PropertySubType
print("Summary by PropertySubType:")
print(sold.groupby('PropertySubType').agg(
    transaction_count=('ClosePrice', 'count'),
    median_close_price=('ClosePrice', 'median'),
    median_price_per_sqft=('price_per_sqft', 'median'),
    median_days_on_market=('DaysOnMarket', 'median'),
    avg_price_ratio=('price_ratio', 'mean')
).sort_values('transaction_count', ascending=False).to_string())

Summary by PropertySubType:
                       transaction_count  median_close_price  median_price_per_sqft  median_days_on_market  avg_price_ratio
PropertySubType                                                                                                            
SingleFamilyResidence             314396            880000.0             526.315789                   18.0         1.018280
Condominium                        70676            625000.0             563.043478                   24.0         0.993283
Townhouse                          24645            795000.0             555.555556                   18.0         1.047561
ManufacturedOnLand                  5519            325000.0             226.116450                   30.0         0.971904
Duplex                              2338            910000.0             543.478261                   21.0         0.990186
StockCooperative                    1692            360000.0             396.000000                   20

In [6]:
# Segment Analysis — Top 15 Listing Offices by volume
print("Top 15 Listing Offices by transaction count:")
print(sold.groupby('ListOfficeName').agg(
    transaction_count=('ClosePrice', 'count'),
    total_volume=('ClosePrice', 'sum'),
    median_close_price=('ClosePrice', 'median')
).sort_values('transaction_count', ascending=False).head(15).to_string())

Top 15 Listing Offices by transaction count:
                                                       transaction_count  total_volume  median_close_price
ListOfficeName                                                                                            
Compass                                                            29210  5.259146e+10           1325000.0
Coldwell Banker Realty                                             18442  2.865479e+10           1160000.0
Keller Williams Realty                                              8369  8.567547e+09            868000.0
First Team Real Estate                                              5886  6.579554e+09            959000.0
Berkshire Hathaway HomeServices California Properties               5580  8.372260e+09            950000.0
Real Broker                                                         4978  5.199508e+09            840000.0
eXp Realty of California Inc                                        4894  4.683777e+09            7

In [7]:
# Segment Analysis — Top 15 Buyer Offices by volume
print("Top 15 Buyer Offices by transaction count:")
print(sold.groupby('BuyerOfficeName').agg(
    transaction_count=('ClosePrice', 'count'),
    total_volume=('ClosePrice', 'sum'),
    median_close_price=('ClosePrice', 'median')
).sort_values('transaction_count', ascending=False).head(15).to_string())

Top 15 Buyer Offices by transaction count:
                                                       transaction_count  total_volume  median_close_price
BuyerOfficeName                                                                                           
Compass                                                            27374  4.823397e+10           1301000.0
Coldwell Banker Realty                                             14785  2.309358e+10           1160000.0
NONMEMBER MRML                                                      9095  5.715968e+09            502189.0
Keller Williams Realty                                              6600  6.746791e+09            830000.0
Real Broker                                                         6453  6.381729e+09            795000.0
eXp Realty of California Inc                                        5332  5.477081e+09            825000.0
First Team Real Estate                                              5325  5.752953e+09            897

In [8]:
# Feature Engineering — Listings Dataset

# 1. Price Reduction Ratio — list price vs original list price
listings['price_reduction_ratio'] = listings['ListPrice'] / listings['OriginalListPrice']

# 2. Price Per Square Foot — normalizes price across different home sizes
listings['price_per_sqft'] = listings['ListPrice'] / listings['LivingArea']

# 3. Listing to Contract Days — time from listing to accepted offer
listings['listing_to_contract_days'] = (
    listings['PurchaseContractDate'] - listings['ListingContractDate']
).dt.days

# 4. Contract to Close Days — escrow and closing period duration
listings['contract_to_close_days'] = (
    listings['CloseDate'] - listings['PurchaseContractDate']
).dt.days

# 5. Time series variables derived from ListingContractDate
listings['listing_year'] = listings['ListingContractDate'].dt.year
listings['listing_month'] = listings['ListingContractDate'].dt.month
listings['listing_yrmo'] = listings['ListingContractDate'].dt.to_period('M')

# Sample output table
print("Sample of engineered metrics (listings):")
listings[['ListPrice', 'OriginalListPrice', 'LivingArea',
          'price_reduction_ratio', 'price_per_sqft',
          'listing_to_contract_days', 'contract_to_close_days',
          'listing_year', 'listing_month', 'listing_yrmo']].head(10)

Sample of engineered metrics (listings):


,ListPrice,OriginalListPrice,LivingArea,price_reduction_ratio,price_per_sqft,listing_to_contract_days,contract_to_close_days,listing_year,listing_month,listing_yrmo
0,149900.0,115900.0,1008.0,1.293356,148.710317,804.0,NaN,2024,1,2024-01
1,1499900.0,1499900.0,1370.0,1.000000,1094.817518,811.0,NaN,2024,1,2024-01
2,2199000.0,2500000.0,2573.0,0.879600,854.644384,785.0,NaN,2024,1,2024-01
3,3200000.0,3200000.0,1381.0,1.000000,2317.161477,58.0,421.0,2024,1,2024-01
4,1795000.0,1795000.0,3200.0,1.000000,560.937500,479.0,27.0,2024,1,2024-01
5,299999.0,319000.0,1310.0,0.940436,229.006870,572.0,59.0,2024,1,2024-01
6,1250000.0,1250000.0,1950.0,1.000000,641.025641,455.0,23.0,2024,1,2024-01
7,4924000.0,4924000.0,3413.0,1.000000,1442.719016,366.0,34.0,2024,1,2024-01
8,550000.0,550000.0,2705.0,1.000000,203.327172,395.0,41.0,2024,1,2024-01
9,535000.0,535000.0,1519.0,1.000000,352.205398,366.0,28.0,2024,1,2024-01


In [9]:
# Save enriched datasets
sold.to_csv('../data/02_intermediate/CRMLSSold_Cleaned.csv', index=False)
print(f"Saved CRMLSSold_Cleaned.csv — {len(sold):,} rows, {sold.shape[1]} columns")

listings.to_csv('../data/02_intermediate/CRMLSListing_Cleaned.csv', index=False)
print(f"Saved CRMLSListing_Cleaned.csv — {len(listings):,} rows, {listings.shape[1]} columns")

Saved CRMLSSold_Cleaned.csv — 421,589 rows, 84 columns
Saved CRMLSListing_Cleaned.csv — 467,728 rows, 76 columns
